In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\Program Files\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
W0419 15:04:03.931000 24460 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
MODEL_NAME = "bert-base-uncased"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"

OUT_DIR = "module6_sentiment_bert"
MAX_LEN = 128

# Keep it fast
TRAIN_SIZE = 2000
VAL_SIZE = 500
TEST_SIZE = 500

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)

train_ds = raw["train"].shuffle(seed=42).select(range(min(TRAIN_SIZE, len(raw["train"]))))
val_ds = raw["validation"].shuffle(seed=42).select(range(min(VAL_SIZE, len(raw["validation"]))))

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 91197.94 examples/s]


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

c:\Program Files\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
def tokenize_batch(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok = val_ds.map(tokenize_batch, batched=True)

# rename label -> labels for Trainer compatibility
train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")

Map: 100%|██████████| 500/500 [00:00<00:00, 13539.45 examples/s]


In [6]:
cols_to_keep = {"input_ids", "attention_mask", "labels", "token_type_ids"}
train_cols = [c for c in train_tok.column_names if c not in cols_to_keep]
val_cols = [c for c in val_tok.column_names if c not in cols_to_keep]

train_tok = train_tok.remove_columns(train_cols)
val_tok = val_tok.remove_columns(val_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# SST-2 labels: 0 = negative, 1 = positive
model.config.id2label = {0: "negative", 1: "positive"}
model.config.label2id = {"negative": 0, "positive": 1}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [9]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

 21%|██        | 26/125 [00:03<00:12,  7.83it/s]

{'loss': 0.6752, 'grad_norm': 7.461766719818115, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.2}


 42%|████▏     | 52/125 [00:06<00:08,  8.75it/s]

{'loss': 0.5399, 'grad_norm': 4.791096210479736, 'learning_rate': 1.2e-05, 'epoch': 0.4}


 62%|██████▏   | 77/125 [00:09<00:05,  8.85it/s]

...

                                                 
100%|██████████| 125/125 [00:15<00:00,  8.40it/s]

{'eval_loss': 0.3270084261894226, 'eval_accuracy': 0.872, 'eval_precision': 0.8773946360153256, 'eval_recall': 0.8773946360153256, 'eval_f1': 0.8773946360153256, 'eval_runtime': 0.8309, 'eval_samples_per_second': 601.742, 'eval_steps_per_second': 19.256, 'epoch': 1.0}


100%|██████████| 125/125 [00:17<00:00,  7.10it/s]

{'train_runtime': 17.6065, 'train_samples_per_second': 113.595, 'train_steps_per_second': 7.1, 'train_loss': 0.4529389305114746, 'epoch': 1.0}


TrainOutput(global_step=125, training_loss=0.4529389305114746, metrics={'train_runtime': 17.6065, 'train_samples_per_second': 113.595, 'train_steps_per_second': 7.1, 'train_loss': 0.4529389305114746, 'epoch': 1.0})

In [11]:
metrics = trainer.evaluate()
print(metrics)

with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

100%|██████████| 16/16 [00:00<00:00, 18.98it/s]

{'eval_loss': 0.3270084261894226, 'eval_accuracy': 0.872, 'eval_precision': 0.8773946360153256, 'eval_recall': 0.8773946360153256, 'eval_f1': 0.8773946360153256, 'eval_runtime': 0.948, 'eval_samples_per_second': 527.399, 'eval_steps_per_second': 16.877, 'epoch': 1.0}


In [12]:
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)

    outputs = []
    for text, pred, prob in zip(texts, preds, probs):
        outputs.append({
            "text": text,
            "prediction": model.config.id2label[int(pred)],
            "negative_prob": float(prob[0]),
            "positive_prob": float(prob[1]),
        })
    return outputs


In [13]:
samples = [
    "I really like this product.",
    "This is the worst experience I have ever had.",
    "The movie was okay, nothing special."
]

In [14]:
results = predict_sentiment(samples)
for r in results:
    print(r)

pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, "sample_predictions.csv"), index=False, encoding="utf-8")

{'text': 'I really like this product.', 'prediction': 'positive', 'negative_prob': 0.061971522867679596, 'positive_prob': 0.9380285143852234}
{'text': 'This is the worst experience I have ever had.', 'prediction': 'negative', 'negative_prob': 0.8490504026412964, 'positive_prob': 0.1509496122598648}
{'text': 'The movie was okay, nothing special.', 'prediction': 'positive', 'negative_prob': 0.3037932217121124, 'positive_prob': 0.6962067484855652}
